# Experimento: tamanho da janela de observação (Random Forest)

Este notebook trata o tamanho da janela de observação (`window`) da
representação tabular — usada por `ssd_utils.build_windowed_features` para
resumir os últimos `window` dias de leitura SMART em
`[atual, média, desvio padrão, mínimo, máximo]` por atributo (24 x 5 = 120
features) — como **hiperparâmetro**, e varre alguns valores para o
**Random Forest**.

### Por que só o Random Forest?

A Regressão Logística é um baseline linear; varrê-la também não traria
informação adicional relevante para a escolha da janela e tem custo alto
(treino lento mesmo com `solver='saga'`). A janela vencedora encontrada aqui
será aplicada também à LR, fora deste notebook.

### Protocolo

- Os hiperparâmetros do Random Forest (`best_params`) são carregados de
  `resultados_random_forest.pkl` (gerado por `AMA_projeto_RandomForest.ipynb`
  com `window=14`) e usados **fixos** para todas as janelas testadas — isso
  viabiliza o custo computacional da varredura. A janela vencedora passará por
  uma busca de hiperparâmetros completa quando o notebook oficial do RF for
  re-executado com ela.
- A seleção da janela é feita **exclusivamente no conjunto de validação**,
  usando **Average Precision (AP)** como critério primário e **F1** (no limiar
  ótimo, `ssd.best_threshold_f1`) como critério secundário.
- O conjunto de **teste não é avaliado neste notebook** — isso preserva o
  teste como avaliação final e não enviesada. A janela vencedora será usada
  nos notebooks oficiais de RF e LR; a re-execução desses notebooks (com busca
  de hiperparâmetros completa) produz o resultado de teste oficial,
  posteriormente consolidado em `AMA_comparacao_modelos.ipynb`.
- `contamination_level = 7` (horizonte do rótulo) é **fixo e não muda** —
  apenas a janela de **features** é variada.

### Por que a varredura para em window = 90?

Janelas muito grandes aproximam as estatísticas móveis (`média`/`mín`/`máx`/
`desvio` dos últimos `window` dias) das estatísticas acumuladas desde o início
do disco (que vive, no máximo, 360 dias) — diluindo o contraste entre "resumo
de janela local" (a representação tabular) e "sequência completa" (o LSTM),
que é exatamente o que este experimento quer medir. Janelas muito maiores
tendem apenas a reproduzir, de forma cada vez mais borrada, o que o LSTM já vê
de forma exata.


In [1]:
# ===== Configuracao =====
import os
import pickle
import time

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

import os
import sys

# --- Resolucao robusta de caminhos (independente do diretorio de trabalho) ---
def _encontra_raiz(inicio=None):
    """Sobe na arvore ate achar comum/ssd_utils.py."""
    d = os.path.abspath(inicio or os.getcwd())
    while True:
        if os.path.isfile(os.path.join(d, 'comum', 'ssd_utils.py')):
            return d
        pai = os.path.dirname(d)
        if pai == d:
            raise RuntimeError('raiz do projeto (com comum/ssd_utils.py) nao encontrada')
        d = pai

PROJ_ROOT = _encontra_raiz()
COMUM_DIR = os.path.join(PROJ_ROOT, 'comum')
H7_DIR = os.path.join(PROJ_ROOT, 'horizonte_7dias')
H30_DIR = os.path.join(PROJ_ROOT, 'horizonte_30dias')
EXP_DIR = os.path.join(PROJ_ROOT, 'experimentos')
if COMUM_DIR not in sys.path:
    sys.path.insert(0, COMUM_DIR)
# ---------------------------------------------------------------------------
import ssd_utils as ssd

DATA_DIR = COMUM_DIR        # data.pickle / mask.pickle (pasta comum/)
OUTPUT_DIR = EXP_DIR        # saidas deste experimento
CONTAMINATION_LEVEL = 7     # horizonte de previsao de falha - fixo, nao alterar
RANDOM_STATE = 42

# ----- Carrega dados, rotulos e splits (identico aos notebooks oficiais) -----
data, mask = ssd.load_dataset(DATA_DIR)
print(f'data: {data.shape}   mask: {mask.shape}')

labels = ssd.create_class_labels(data, mask, CONTAMINATION_LEVEL)
split_idx = ssd.split_indices(n_discos=data.shape[0])

# ----- Hiperparametros do RF (busca feita com window=14 em AMA_projeto_RandomForest.ipynb) -----
RF_RESULTS_PATH = os.path.join(H7_DIR, 'resultados_random_forest.pkl')
if not os.path.exists(RF_RESULTS_PATH):
    raise FileNotFoundError(
        f'{RF_RESULTS_PATH} nao encontrado - execute AMA_projeto_RandomForest.ipynb '
        'primeiro (ele gera este arquivo com a busca de hiperparametros para window=14).'
    )

with open(RF_RESULTS_PATH, 'rb') as f:
    rf_results = pickle.load(f)

best_params = rf_results['best_params']
print(f'Hiperparametros do RF (window=14): {best_params}')
print('Usados fixos para todas as janelas desta varredura - viabiliza o custo '
      'computacional. A janela vencedora passara por busca completa quando o '
      'notebook oficial do RF for re-executado com ela.')

data: (5343, 360, 24)   mask: (5343, 360)
Hiperparametros do RF (window=14): {'n_estimators': 100, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': None}
Usados fixos para todas as janelas desta varredura - viabiliza o custo computacional. A janela vencedora passara por busca completa quando o notebook oficial do RF for re-executado com ela.


In [2]:
WINDOWS = [7, 14, 30, 60, 90]
resultados = []

for w in WINDOWS:
    print(f'===== window = {w} =====')
    # reconstroi as features (sobrescreve as variaveis a cada iteracao
    # para nao acumular ~900 MB por janela na memoria)
    X_all, disco_id, timestep, feature_cols = ssd.build_windowed_features(
        data, mask, window=w)
    y_flat = labels.reshape(-1)
    mask_flat = mask.reshape(-1)

    splits = {nome: ssd.select_split(X_all, disco_id, timestep, y_flat,
                                      mask_flat, idx)
              for nome, idx in split_idx.items()}
    del X_all

    rf = RandomForestClassifier(**best_params, class_weight='balanced',
                                 random_state=RANDOM_STATE, n_jobs=-1)
    t0 = time.time()
    rf.fit(splits['train']['X'], splits['train']['y'])
    fit_time = time.time() - t0

    val_proba = rf.predict_proba(splits['validation']['X'])[:, 1]
    thr = ssd.best_threshold_f1(splits['validation']['y'], val_proba)
    val_pred = (val_proba >= thr).astype(int)

    m = ssd.evaluate_predictions(splits['validation']['y'], val_pred)
    ap = average_precision_score(splits['validation']['y'], val_proba)
    auc = roc_auc_score(splits['validation']['y'], val_proba)

    resultados.append({'window': w, 'ap_val': ap, 'auc_val': auc,
                        'f1_val': m['f1_score'], 'precision_val': m['precision'],
                        'recall_val': m['recall'], 'threshold': thr,
                        'fit_time_s': fit_time})
    print(f"AP={ap:.4f}  AUC={auc:.4f}  F1={m['f1_score']:.4f}  thr={thr:.4f}  "
          f"fit={fit_time:.0f}s")
    del splits, rf

===== window = 7 =====


AP=0.1624  AUC=0.7714  F1=0.2339  thr=0.3899  fit=48s
===== window = 14 =====


AP=0.1576  AUC=0.7705  F1=0.2337  thr=0.3777  fit=46s
===== window = 30 =====


AP=0.1543  AUC=0.7692  F1=0.2322  thr=0.3528  fit=45s
===== window = 60 =====


AP=0.1616  AUC=0.7868  F1=0.2412  thr=0.3593  fit=44s
===== window = 90 =====


AP=0.1650  AUC=0.7973  F1=0.2378  thr=0.4132  fit=44s


In [3]:
df_resultados = pd.DataFrame(resultados).sort_values('ap_val', ascending=False).reset_index(drop=True)

header = (f"{'window':>8} {'AP(val)':>10} {'AUC(val)':>10} {'F1(val)':>10} "
          f"{'Prec(val)':>10} {'Rec(val)':>10} {'limiar':>8} {'fit(s)':>8}")
print(header)
print('-' * len(header))
for i, row in df_resultados.iterrows():
    marcador = '  <-- melhor (AP)' if i == 0 else ''
    print(f"{int(row['window']):>8} {row['ap_val']:>10.4f} {row['auc_val']:>10.4f} "
          f"{row['f1_val']:>10.4f} {row['precision_val']:>10.4f} {row['recall_val']:>10.4f} "
          f"{row['threshold']:>8.4f} {row['fit_time_s']:>8.1f}{marcador}")

melhor_window = int(df_resultados.iloc[0]['window'])
print(f'\nJanela vencedora (maior AP na validacao): window = {melhor_window}')

ssd.save_results(os.path.join(OUTPUT_DIR, 'experimento_janela.pkl'),
                  resultados=resultados, best_window=melhor_window,
                  rf_best_params=best_params)

  window    AP(val)   AUC(val)    F1(val)  Prec(val)   Rec(val)   limiar   fit(s)
---------------------------------------------------------------------------------
      90     0.1650     0.7973     0.2378     0.2537     0.2237   0.4132     43.8  <-- melhor (AP)
       7     0.1624     0.7714     0.2339     0.2314     0.2366   0.3899     47.5
      60     0.1616     0.7868     0.2412     0.2167     0.2719   0.3593     44.2
      14     0.1576     0.7705     0.2337     0.2435     0.2246   0.3777     45.9
      30     0.1543     0.7692     0.2322     0.2072     0.2641   0.3528     44.9

Janela vencedora (maior AP na validacao): window = 90
Resultados salvos em: C:\Users\len108\Documents\pv\ErroSSD\experimentos\experimento_janela.pkl


In [4]:
windows = [r['window'] for r in resultados]
ap_vals = [r['ap_val'] for r in resultados]
f1_vals = [r['f1_val'] for r in resultados]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle('Random Forest - Varredura do tamanho da janela (conjunto de validacao)', fontsize=12)

for ax, vals, titulo, cor in zip(axes, [ap_vals, f1_vals],
                                  ['Average Precision (AP)', 'F1-score (limiar otimo)'],
                                  ['#1565C0', '#E65100']):
    ax.plot(windows, vals, marker='o', color=cor, lw=2)
    ax.set_xscale('log', base=2)
    ax.set_xticks(windows)
    ax.set_xticklabels([str(w) for w in windows])
    ax.set_xlabel('Tamanho da janela (dias)')
    ax.set_ylabel(titulo)
    ax.set_title(titulo)
    ax.grid(alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'exp_janela.png'), bbox_inches='tight')
plt.show()
print('Figura salva em exp_janela.png')

Figura salva em exp_janela.png


C:\Users\len108\AppData\Local\Temp\ipykernel_14580\931631759.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Próximos passos

1. Anote a janela vencedora (maior AP na validação) impressa acima.
2. Atualize `WINDOW = <vencedora>` em `AMA_projeto_RandomForest.ipynb` e em
   `AMA_projeto_LogisticRegression.ipynb`.
3. Re-execute os dois notebooks — a busca de hiperparâmetros
   (`RandomizedSearchCV`) será refeita do zero para a nova janela, encontrando
   os melhores hiperparâmetros para essa representação (os `best_params`
   usados aqui foram apenas os da janela 14, fixos por custo computacional).
4. Re-execute `AMA_comparacao_modelos.ipynb` para gerar a comparação final
   (LSTM x LR x RF) com os resultados de teste oficiais.
